<a href="https://colab.research.google.com/github/ananyamk26/AI-LAB/blob/main/1WA24CS040_Week9_FOL_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import re

# Tokenizer
def tokenize(s):
    pattern = r'<=>|=>|[~&|(),]|forall|exists|[A-Za-z_]\w*'
    tokens = re.findall(pattern, s)
    if ''.join(tokens).replace(' ', '') != re.sub(r'\s+', '', s):
        raise ValueError("Invalid expression")
    return tokens


# Parser
class Parser:
    def __init__(self, tokens):
        self.t = tokens
        self.i = 0

    def peek(self):
        return self.t[self.i] if self.i < len(self.t) else None

    def take(self):
        x = self.peek()
        if x is None:
            raise ValueError("Unexpected end")
        self.i += 1
        return x

    def parse(self):
        result = self.iff()
        if self.peek() is not None:
            raise ValueError("Unexpected token: " + self.peek())
        return result

    def iff(self):
        a = self.imp()
        while self.peek() == '<=>':
            self.take()
            a = ('iff', a, self.imp())
        return a

    def imp(self):
        a = self.disj()
        if self.peek() == '=>':
            self.take()
            return ('imp', a, self.imp())
        return a

    def disj(self):
        a = self.conj()
        while self.peek() == '|':
            self.take()
            a = ('or', a, self.conj())
        return a

    def conj(self):
        a = self.unary()
        while self.peek() == '&':
            self.take()
            a = ('and', a, self.unary())
        return a

    def unary(self):
        x = self.peek()

        if x == '~':
            self.take()
            return ('not', self.unary())

        if x in ('forall', 'exists'):
            q = self.take()
            v = self.take()
            return (q, v, self.unary())

        if x == '(':
            self.take()
            a = self.iff()
            if self.take() != ')':
                raise ValueError("Missing closing bracket")
            return a

        name = self.take()
        args = []

        if self.peek() == '(':
            self.take()
            if self.peek() != ')':
                args.append(self.take())
                while self.peek() == ',':
                    self.take()
                    args.append(self.take())
            if self.take() != ')':
                raise ValueError("Missing closing bracket")

        return ('pred', name, args)


# Display expression
def show(a):
    op = a[0]

    if op == 'pred':
        return a[1] + ('(' + ','.join(a[2]) + ')' if a[2] else '')

    if op == 'not':
        return '~' + show(a[1])

    if op in ('forall', 'exists'):
        return op + ' ' + a[1] + ' (' + show(a[2]) + ')'

    symbols = {
        'and': '&', 'or': '|',
        'imp': '=>', 'iff': '<=>'
    }

    return '(' + show(a[1]) + ' ' + symbols[op] + ' ' + show(a[2]) + ')'


# Substitute variables
def substitute(a, old, new):
    op = a[0]

    if op == 'pred':
        return ('pred', a[1],
                [new if x == old else x for x in a[2]])

    if op == 'not':
        return ('not', substitute(a[1], old, new))

    if op in ('forall', 'exists'):
        if a[1] == old:
            return a
        return (op, a[1], substitute(a[2], old, new))

    return (op, substitute(a[1], old, new),
            substitute(a[2], old, new))


# Step 1: Eliminate implications and biconditionals
def eliminate(a):
    op = a[0]

    if op == 'iff':
        p = eliminate(a[1])
        q = eliminate(a[2])
        return ('and',
                ('or', ('not', p), q),
                ('or', ('not', q), p))

    if op == 'imp':
        return ('or', ('not', eliminate(a[1])),
                eliminate(a[2]))

    if op in ('pred',):
        return a

    if op in ('forall', 'exists'):
        return (op, a[1], eliminate(a[2]))

    if op == 'not':
        return ('not', eliminate(a[1]))

    return (op, eliminate(a[1]), eliminate(a[2]))


# Step 2: Move negations inward
def nnf(a):
    op = a[0]

    if op == 'not':
        x = a[1]
        if x[0] == 'not':
            return nnf(x[1])
        if x[0] == 'and':
            return ('or', nnf(('not', x[1])),
                    nnf(('not', x[2])))
        if x[0] == 'or':
            return ('and', nnf(('not', x[1])),
                    nnf(('not', x[2])))
        if x[0] == 'forall':
            return ('exists', x[1], nnf(('not', x[2])))
        if x[0] == 'exists':
            return ('forall', x[1], nnf(('not', x[2])))
        return a

    if op in ('forall', 'exists'):
        return (op, a[1], nnf(a[2]))

    if op in ('and', 'or'):
        return (op, nnf(a[1]), nnf(a[2]))

    return a


# Step 3: Standardize variables
def standardize(a):
    counter = [0]

    def rename(node, env):
        op = node[0]

        if op == 'pred':
            return ('pred', node[1],
                    [env.get(x, x) for x in node[2]])

        if op == 'not':
            return ('not', rename(node[1], env))

        if op in ('forall', 'exists'):
            counter[0] += 1
            old = node[1]
            new = old + '_' + str(counter[0])
            new_env = env.copy()
            new_env[old] = new
            return (op, new, rename(node[2], new_env))

        return (op, rename(node[1], env),
                rename(node[2], env))

    return rename(a, {})


# Step 4: Skolemization
def skolemize(a):
    counter = [0]

    def convert(node, universals):
        op = node[0]

        if op == 'exists':
            counter[0] += 1
            sk = 'SK' + str(counter[0])

            if universals:
                sk += '(' + ','.join(universals) + ')'

            body = substitute(node[2], node[1], sk)
            return convert(body, universals)

        if op == 'forall':
            return convert(node[2], universals + [node[1]])

        if op in ('and', 'or'):
            return (op, convert(node[1], universals),
                    convert(node[2], universals))

        if op == 'not':
            return ('not', convert(node[1], universals))

        return node

    return convert(a, [])


# Step 5: Remove universal quantifiers
def remove_universal(a):
    if a[0] == 'forall':
        return remove_universal(a[2])

    if a[0] in ('and', 'or'):
        return (a[0], remove_universal(a[1]),
                remove_universal(a[2]))

    return a


# Step 6: Distribute OR over AND
def distribute(a, b):
    if a[0] == 'and':
        return ('and',
                distribute(a[1], b),
                distribute(a[2], b))

    if b[0] == 'and':
        return ('and',
                distribute(a, b[1]),
                distribute(a, b[2]))

    return ('or', a, b)


def cnf(a):
    if a[0] in ('and', 'or'):
        left = cnf(a[1])
        right = cnf(a[2])

        if a[0] == 'and':
            return ('and', left, right)

        return distribute(left, right)

    return a


# Main CNF conversion
def convert_to_cnf(expression):
    print("\nOriginal Expression:", expression)

    tokens = tokenize(expression)
    result = Parser(tokens).parse()

    result = eliminate(result)
    print("\n1. Eliminate implications:", show(result))

    result = nnf(result)
    print("\n2. Move negations inward:", show(result))

    result = standardize(result)
    print("\n3. Standardize variables:", show(result))

    result = skolemize(result)
    print("\n4. Skolemization:", show(result))

    result = remove_universal(result)
    print("\n5. Remove universal quantifiers:", show(result))

    result = cnf(result)
    print("\n6. Final CNF:", show(result))

    return result


# Main program
print("FIRST ORDER LOGIC TO CNF CONVERTER")
print("----------------------------------")

expression = input("Enter FOL expression: ")

try:
    result = convert_to_cnf(expression)
    print("\nCNF Result:", show(result))
except (ValueError, IndexError) as e:
    print("Error:", e)


FIRST ORDER LOGIC TO CNF CONVERTER
----------------------------------
Enter FOL expression:  forall x (P(x) => exists y Q(x,y))

Original Expression:  forall x (P(x) => exists y Q(x,y))

1. Eliminate implications: forall x ((~P(x) | exists y (Q(x,y))))

2. Move negations inward: forall x ((~P(x) | exists y (Q(x,y))))

3. Standardize variables: forall x_1 ((~P(x_1) | exists y_2 (Q(x_1,y_2))))

4. Skolemization: (~P(x_1) | Q(x_1,SK1(x_1)))

5. Remove universal quantifiers: (~P(x_1) | Q(x_1,SK1(x_1)))

6. Final CNF: (~P(x_1) | Q(x_1,SK1(x_1)))

CNF Result: (~P(x_1) | Q(x_1,SK1(x_1)))
